# PyTorch Tensör Boyut Manipülasyonu

Ödev: (32, 10, 64) boyutunda bir tensör oluşturup (320, 64) haline getirmek,
(64, 128) bir ağırlık matrisiyle çarpmak, sonra tekrar (32, 10, 128) yapmak.
En sonda da view ile reshape arasındaki farkı anlatmak.

Yapacağım işlem şu şekilde ilerliyor:

(32, 10, 64) -> (320, 64) -> (64, 128) ile çarpım -> (320, 128) -> (32, 10, 128)

Her adımdan sonra shape yazdırdım.

In [1]:
import torch

# her çalıştırdığımda aynı sayılar gelsin diye seed veriyorum
torch.manual_seed(42)

print(torch.__version__)

2.5.1+cu121


## 1. Adım - Tensörü oluşturuyorum

batch = 32 -> aynı anda işlenen cümle sayısı
sequence = 10 -> her cümlede 10 token var
embedding = 64 -> her token 64 boyutlu bir vektörle temsil ediliyor

Yani 32 tane cümlem var, her cümlede 10 kelime, her kelime 64 sayıdan oluşan bir vektör.

In [2]:
batch, sequence, embedding = 32, 10, 64

x = torch.randn(batch, sequence, embedding)
print(x.shape)

torch.Size([32, 10, 64])


## 2. Adım - (320, 64) haline getiriyorum

Matris çarpımı 2 boyutlu çalıştığı için batch ve sequence eksenlerini birleştirdim,
32 * 10 = 320 oldu. Artık tensör "32 cümlenin 10 tokeni" değil de düz 320 token gibi
duruyor. Embedding ekseni (64) bozulmadığı için her tokenin kendi vektörü aynı kalıyor.

-1 yazınca PyTorch o boyutu kendi hesaplıyor, 320'yi elle yazmama gerek kalmıyor.

In [3]:
x_flat = x.view(-1, embedding)   # 32 * 10 = 320
print(x_flat.shape)

torch.Size([320, 64])


In [4]:
# aynı şeyi reshape ile de yapabiliyorum, farkı en sonda anlattım
x_flat_reshape = x.reshape(320, 64)
print(x_flat_reshape.shape)

print("sonuçlar aynı mı:", torch.equal(x_flat, x_flat_reshape))

# view kopya çıkarmıyor, x ile aynı belleği kullanıyor
print("aynı bellek mi:", x.data_ptr() == x_flat.data_ptr())

torch.Size([320, 64])
sonuçlar aynı mı: True
aynı bellek mi: True


## 3. Adım - Ağırlık matrisi

64 boyutlu tokenleri 128 boyuta çıkaracak bir matris. Çarpımın çalışması için
soldaki tensörün son boyutu (64) ile bu matrisin ilk boyutu (64) aynı olmak zorunda.

In [5]:
out_features = 128

weight = torch.randn(embedding, out_features)
print(weight.shape)

torch.Size([64, 128])


## 4. Adım - Çarpma işlemi

(320, 64) @ (64, 128) yapınca ortadaki 64'ler gidiyor, geriye (320, 128) kalıyor.
@ operatörü torch.matmul ile aynı şey.

In [6]:
out_flat = x_flat @ weight
print(out_flat.shape)

torch.Size([320, 128])


## 5. Adım - Tekrar 3 boyuta çeviriyorum

Çarpım bitti, şimdi 320 tokeni tekrar 32 cümle x 10 token şeklinde ayırıyorum ki
çıktı yine batch yapısında olsun ve bir sonraki katmana verilebilsin.

In [7]:
out = out_flat.view(batch, sequence, out_features)
print(out.shape)

torch.Size([32, 10, 128])


### Kontrol

PyTorch aslında 3 boyutlu tensörü direkt de çarpabiliyor. Merak ettim, elle yaptığım
düzleştir - çarp - geri çevir işleminin sonucuyla aynı mı diye kontrol ettim.

In [8]:
out_direct = x @ weight
print(out_direct.shape)

print("iki sonuç aynı mı:", torch.allclose(out, out_direct, atol=1e-5))

torch.Size([32, 10, 128])
iki sonuç aynı mı: True


### Adımların özeti

In [9]:
print("1) giris tensoru       :", tuple(x.shape))
print("2) duzlestirilmis      :", tuple(x_flat.shape))
print("3) agirlik matrisi     :", tuple(weight.shape))
print("4) carpim sonucu       :", tuple(out_flat.shape))
print("5) geri cevrilmis      :", tuple(out.shape))

1) giris tensoru       : (32, 10, 64)
2) duzlestirilmis      : (320, 64)
3) agirlik matrisi     : (64, 128)
4) carpim sonucu       : (320, 128)
5) geri cevrilmis      : (32, 10, 128)


---

## view ile reshape arasındaki fark

Bu ikisi neden ayrı ayrı var diye araştırdım, anladığım kadarıyla şöyle:

### Tensör bellekte nasıl duruyor

Tensör bellekte aslında tek boyutlu düz bir dizi olarak tutuluyor. (32, 10, 64) gibi
bir şekil sadece bizim ona nasıl baktığımızla ilgili. PyTorch shape'in yanında bir de
stride tutuyor ve "i, j, k indeksindeki eleman düz dizide kaçıncı sırada" sorusunu
bununla hesaplıyor.

Eğer elemanlar bellekte satır satır, sırayla ve boşluksuz duruyorsa o tensöre
contiguous deniyor. torch.randn ile yeni oluşturduğum tensörler contiguous oluyor.

### Farkları

view veriyi hiç kopyalamıyor, sadece shape ve stride bilgisini değiştirip aynı belleğe
farklı bir açıdan bakıyor. Bu yüzden çok ucuz bir işlem ama tensörün contiguous olmasını
şart koşuyor, değilse hata veriyor.

reshape ise önce view yapmayı deniyor, olursa kopyasız dönüyor. Olmuyorsa (tensör
contiguous değilse) veriyi sessizce kopyalayıp düzeltiyor ve öyle şekillendiriyor.
Yani her zaman çalışıyor ama bazen farkında olmadan kopyalama maliyeti çıkarıyor.

### Contiguous ne zaman bozuluyor

transpose, permute, .T gibi işlemler veriyi bellekte gerçekten taşımıyor, sadece
stride'ları değiştiriyor. O yüzden mantıksal sıra ile bellekteki gerçek sıra birbirinden
ayrılıyor ve tensör contiguous olmaktan çıkıyor.

Transformer'larda multi head attention kısmında sürekli permute/transpose yapıldığı için
bu durumla çok karşılaşılıyor, kodlarda gördüğüm .contiguous().view(...) kalıbının sebebi
de buymuş.

### Hangisini kullanmalı

Kopya çıkmadığından emin olmak istiyorsam ve bir sorun varsa sessizce yavaşlamak yerine
hata almak istiyorsam view, tensörün bellekteki durumundan emin değilsem ve kod her
şartta çalışsın istiyorsam reshape.

Aşağıda bu farkı deneyerek gösterdim.

In [10]:
# x contiguous, o yüzden view sorunsuz çalışıyordu
print("x contiguous mu   :", x.is_contiguous())
print("x stride          :", x.stride())

# transpose veriyi taşımıyor, sadece stride değişiyor
x_t = x.transpose(0, 1)
print()
print("x_t shape         :", tuple(x_t.shape))
print("x_t contiguous mu :", x_t.is_contiguous())
print("x_t stride        :", x_t.stride())
print("ayni bellek mi    :", x.data_ptr() == x_t.data_ptr())

x contiguous mu   : True
x stride          : (640, 64, 1)

x_t shape         : (10, 32, 64)
x_t contiguous mu : False
x_t stride        : (64, 640, 1)
ayni bellek mi    : True


In [11]:
# contiguous olmayan tensörde view hata veriyor
try:
    x_t.view(320, 64)
except RuntimeError as e:
    print("hata:", e)

hata: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.


In [12]:
# reshape aynı işi yapabiliyor, çünkü gerektiği için kopyalıyor
x_t_reshaped = x_t.reshape(320, 64)
print("reshape sonucu    :", tuple(x_t_reshaped.shape))
print("kopya cikti mi    :", x_t.data_ptr() != x_t_reshaped.data_ptr())

# view kullanmak istersem önce contiguous çağırmam gerekiyor
x_t_view = x_t.contiguous().view(320, 64)
print("contiguous + view :", tuple(x_t_view.shape))
print("ikisi esit mi     :", torch.equal(x_t_reshaped, x_t_view))

reshape sonucu    : (320, 64)
kopya cikti mi    : True
contiguous + view : (320, 64)
ikisi esit mi     : True


In [13]:
# tensör zaten contiguous ise reshape kopya çıkarmıyor, view gibi davranıyor
y = x.reshape(320, 64)
print("kopya cikti mi    :", x.data_ptr() != y.data_ptr())

kopya cikti mi    : False


---

## Sonuç

(32, 10, 64) tensörünü (320, 64) yapıp (64, 128) matrisle çarptım, sonucu tekrar
(32, 10, 128) haline getirdim. Çıkan sonucun direkt 3 boyutlu çarpımla (x @ weight)
aynı olduğunu da kontrol ettim, aynı çıktı.

view kopya çıkarmıyor ama tensörün contiguous olmasını istiyor, reshape ise mümkünse
view gibi davranıp gerekirse kopyalayarak her durumda çalışıyor. Aradaki fark tensörün
bellekte nasıl durduğuyla ilgili.